## Experiment 11 -- ResNet18 with Differnet Seeds and Differnet Augmentation



In [69]:

import sys
from pathlib import Path
from torchvision import datasets
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import math
from collections import Counter
import pandas as pd

from torch.optim import Adam
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import classification_report

# Project root
PROJECT_DIR = Path.cwd().parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

sys.path.append("../src")

from data import (
    letterbox,
    create_train_val_datasets,
    create_dataloaders
)

from models import SimpleCNN

from train import train_model

from utils import (
    get_predictions,
    plot_confusion_matrix,
    get_classification_report,
    plot_training_history,
    set_seed
)

In [ ]:
NUM_CLASSES = 8
BATCH_SIZE = 32


EPOCHS_FINETUNE = 150


# ============================================================
# GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA/GPU is not available. "
        "Install a CUDA-enabled PyTorch version and make sure the NVIDIA driver is installed."
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

# Enable cuDNN optimizations for GPU training
torch.backends.cudnn.benchmark = True


Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### Paths

In [72]:
MERGED_TRAIN_DIR = PROJECT_DIR / "dataset" / "merged_train"


print("Training directory:", MERGED_TRAIN_DIR)


print("Training directory exists:", MERGED_TRAIN_DIR.exists())

Training directory: c:\Users\Amir\Downloads\project\project\dataset\merged_train
Training directory exists: True


### Transforms

In [ ]:
eval_transform = transforms.Compose([
    transforms.Lambda(
        lambda img: letterbox(
            img,
            target_size=(224, 224)
        )
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


### SEED Configs

In [76]:
SEED_configs = {
    "seed 7" : 7,
    "seed 10": 10 ,
    "seed 30" : 30,
    "seed 42": 42,
    "seed 86": 86,
}

SEED_configs

{'seed 7': 7, 'seed 10': 10, 'seed 30': 30, 'seed 42': 42, 'seed 86': 86}

In [77]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]

IMAGENET_STD = [0.229, 0.224, 0.225]


augmentation_list = {

    "Basic": transforms.Compose([

        transforms.Resize((224, 224)),

        transforms.RandomHorizontalFlip(p=0.5),

        transforms.RandomRotation(degrees=10),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        )
    ]),


    "Color": transforms.Compose([

        transforms.Resize((224, 224)),

        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        )
    ]),


    "Crop + Flip": transforms.Compose([

        transforms.Resize((224, 224)),

        transforms.RandomResizedCrop(
            size=(224, 224),
            scale=(0.7, 1.0)
        ),

        transforms.RandomHorizontalFlip(p=0.5),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        )
    ]),


    "Rotation + Noise": transforms.Compose([

        transforms.Resize((224, 224)),

        transforms.RandomRotation(degrees=10),

        transforms.ToTensor(),

        transforms.Lambda(
            lambda x: torch.clamp(
                x + torch.randn_like(x) * 0.05,
                0,
                1
            )
        ),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        )
    ]),


    "Blur + Erasing": transforms.Compose([

        transforms.Resize((224, 224)),

        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=3)],
            p=0.3
        ),

        transforms.ToTensor(),

        transforms.RandomErasing(
            p=0.3,
            scale=(0.02, 0.15)
        ),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        )
    ]),


    "Full": transforms.Compose([

        transforms.Resize((224, 224)),

        transforms.RandomHorizontalFlip(p=0.5),

        transforms.RandomRotation(degrees=10),

        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2
        ),

        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=3)],
            p=0.3
        ),

        transforms.ToTensor(),

        transforms.RandomErasing(
            p=0.2,
            scale=(0.02, 0.08)
        ),

        transforms.Normalize(
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD
        )
    ])
}


augmentation_count = {

    "Basic": 2,
    "Color": 1,
    "Crop + Flip": 2,
    "Rotation + Noise": 2,
    "Blur + Erasing": 3,
    "Full": 6
}

In [78]:
def run_experiment(name, augmentation_name, seed_value, train_transform):

    print("=" * 60)
    print(f"Experiment: {name}")
    print(f"Augmentation: {augmentation_name}")
    print(f"SEED: {seed_value}")
    print("=" * 60)

    set_seed(seed_value)


    train_dataset, val_dataset, train_indices, val_indices, class_to_idx = \
        create_train_val_datasets(
            data_dir=MERGED_TRAIN_DIR,
            train_transform=train_transform,
            val_transform=eval_transform,
            test_size=0.2,
            seed=42
        )


    train_loader, val_loader = create_dataloaders(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        batch_size=BATCH_SIZE,
        shuffle_train=True
    )


    resnet_full = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )


    resnet_full.fc = nn.Linear(
        resnet_full.fc.in_features,
        NUM_CLASSES
    )


    resnet_full = resnet_full.to(DEVICE)


    for param in resnet_full.parameters():
        param.requires_grad = True


    BACKBONE_LR = 1e-4
    HEAD_LR = 1e-4


    backbone_params = []
    head_params = []


    for param_name, param in resnet_full.named_parameters():

        if param_name.startswith("fc."):
            head_params.append(param)

        else:
            backbone_params.append(param)


    optimizer_full = torch.optim.Adam(
        [
            {
                "params": backbone_params,
                "lr": BACKBONE_LR
            },
            {
                "params": head_params,
                "lr": HEAD_LR
            }
        ]
    )


    criterion_full = nn.CrossEntropyLoss()


    config_full = {
        "model": "ResNet18",
        "weights": "ImageNet pretrained",
        "training_mode": "full_fine_tuning",
        "augmentation": augmentation_name,
        "optimizer": "Adam",
        "backbone_lr": BACKBONE_LR,
        "head_lr": HEAD_LR,
        "loss": "CrossEntropyLoss",
        "num_classes": NUM_CLASSES,
        "epochs": EPOCHS_FINETUNE,
        "seed": seed_value
    }


    history_full, best_val_acc_full = train_model(
        model=resnet_full,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion_full,
        optimizer=optimizer_full,
        device=DEVICE,
        epochs=EPOCHS_FINETUNE,
        model_name=f"{augmentation_name}_{name}",
        class_to_idx=class_to_idx,
        seed=seed_value,
        config=config_full,
        scheduler=None,
        early_stopping=True,
        patience=50,
        min_delta=0
    )


    return resnet_full, history_full, best_val_acc_full

In [79]:
histories = {}
best_results = {}
results = []

for augmentation_name, transform in augmentation_list.items():

    for name, seed_value in SEED_configs.items():

        experiment_name = f"{augmentation_name}_{name}"

        model, history, best_val_accuracy = run_experiment(
            name=name,
            augmentation_name=augmentation_name,
            seed_value=seed_value,
            train_transform=transform
        )

        histories[experiment_name] = history

        best_results[experiment_name] = best_val_accuracy

        best_epoch = np.argmax(
            history["val_accuracy"]
        ) + 1

        results.append({
            "augmentation": augmentation_name,
            "seed": seed_value,
            "best_epoch": best_epoch,
            "best_val_accuracy": best_val_accuracy
        })

Experiment: seed 7
Augmentation: Basic
SEED: 7
Epoch [1/150] Train Loss: 0.6153 --- Train Acc: 0.8123 --- Val Loss: 0.2540 --- Val Acc: 0.9293 --- LR: 0.000100
  → Validation accuracy improved. Checkpoint saved.
Epoch [2/150] Train Loss: 0.1519 --- Train Acc: 0.9491 --- Val Loss: 0.1577 --- Val Acc: 0.9602 --- LR: 0.000100
  → Validation accuracy improved. Checkpoint saved.
Epoch [3/150] Train Loss: 0.1017 --- Train Acc: 0.9742 --- Val Loss: 0.1670 --- Val Acc: 0.9529 --- LR: 0.000100
  → No improvement (1/50)
Epoch [4/150] Train Loss: 0.0579 --- Train Acc: 0.9849 --- Val Loss: 0.1374 --- Val Acc: 0.9617 --- LR: 0.000100
  → Validation accuracy improved. Checkpoint saved.
Epoch [5/150] Train Loss: 0.0326 --- Train Acc: 0.9923 --- Val Loss: 0.1402 --- Val Acc: 0.9647 --- LR: 0.000100
  → Validation accuracy improved. Checkpoint saved.
Epoch [6/150] Train Loss: 0.0201 --- Train Acc: 0.9948 --- Val Loss: 0.1728 --- Val Acc: 0.9602 --- LR: 0.000100
  → No improvement (1/50)
Epoch [7/150] T

### Trying Differnet SEED Values

In [80]:
results_df = pd.DataFrame(results)

results_df

,augmentation,seed,best_epoch,best_val_accuracy
0,Basic,7,71,0.982327
1,Basic,10,43,0.974963
2,Basic,30,6,0.973490
3,Basic,42,66,0.979381
4,Basic,86,32,0.972018
5,Color,7,74,0.974963
6,Color,10,103,0.967599
7,Color,30,32,0.969072
8,Color,42,73,0.970545
9,Color,86,31,0.964654
